In [23]:
!pip install pyspark

In [24]:
import time

# Give ngrok some more time to establish the tunnel if it hasn't already
time.sleep(5)

# Retrieve and print the public URL for the Spark UI
ngrok_url = !curl -s http://localhost:4040/api/tunnels | grep -oP '(?<="public_url":")[^"]*'

if ngrok_url:
    print(f"Spark UI is available at: {ngrok_url[0]}")
else:
    print("Could not retrieve ngrok public URL. Please ensure ngrok is running.")

Could not retrieve ngrok public URL. Please ensure ngrok is running.


In [25]:
!wget -qnc https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
!unzip -n -q ngrok-stable-linux-amd64.zip

In [26]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
      .config('spark.ui.port', '4040')
      .appName("SparkUI Introdução")
      .getOrCreate()
)

In [27]:
!./ngrok authtoken
get_ipython().system_raw('./ngrok http 4040 &')
!sleep 10
!curl -s http://localhost:4040/api/tunnels | grep -Po 'public_url":"(?=https)\K[^"]*'

NAME:
   authtoken - save authtoken to configuration file

USAGE:
   ngrok authtoken [command options] [arguments...]

DESCRIPTION:
   The authtoken command modifies your configuration file to include
   the specified authtoken. By default, this configuration file is located
   at $HOME/.ngrok2/ngrok.yml

   The ngrok.com service requires that you sign up for an account to use
   many advanced service features. In order to associate your client with
   an account, it must pass a secret token to the ngrok.com service when it
   starts up. Instead of passing this authtoken on every invocation, you may
   use this command to save it into your configuration file so that your
   client always authenticates you properly.

EXAMPLE:
    ngrok authtoken BDZIXnhJt2HNWLXyQ5PM_qCaBq0W2sNFcCa0rfTZd

OPTIONS:
   --config 		save in this config file, default: ~/.ngrok2/ngrok.yml
   --log "false"	path to log file, 'stdout', 'stderr' or 'false'
   --log-format "term"	log record format: 'term', 'logfmt',

In [28]:
# {
#     "id_transacao": 1000,
#     "valor": "58931.97",
#     "remetente": {"nome": "Jonathan Gonsalves", "banco": "BTG", "tipo": "PF"},
#     "destinatario": {"nome": "Emanuella Moura", "banco": "Itau", "tipo": "PJ"},
#     "transaction_date": "2021-06-02",
#     "chave_pix": "aleatoria",
#     "fraude": "1"
# }

from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, TimestampType

schema_remetente_destinatario = StructType([
    StructField('nome', StringType()),
    StructField('banco', StringType()),
    StructField('tipo', StringType()),
])


schema_base_pix = StructType([
    StructField('id_transacao', IntegerType()),
    StructField('valor', DoubleType()),
    StructField('remetente', schema_remetente_destinatario),
    StructField('destinatario', schema_remetente_destinatario),
    StructField('transaction_date', TimestampType()),
    StructField('chave_pix', StringType()),
    StructField('fraude', IntegerType())
])


caminho_json = '/content/pix_transactions.json'

df = spark.read.json(
    caminho_json,
    schema=schema_base_pix,
    timestampFormat="yyyy-MM-dd"
)

In [29]:
df.show()

+------------+--------+--------------------+--------------------+-------------------+---------+------+
|id_transacao|   valor|           remetente|        destinatario|   transaction_date|chave_pix|fraude|
+------------+--------+--------------------+--------------------+-------------------+---------+------+
|        1000|    7.05|{Jonathan Gonsalv...|{Gabriel Cunha, I...|2022-03-19 00:00:00|      cpf|     0|
|        1001|   37.28|{Jonathan Gonsalv...|{Diego Souza, XP,...|2021-01-26 00:00:00|aleatoria|     0|
|        1002|  282.73|{Jonathan Gonsalv...|{Nicole Nunes, BT...|2022-05-31 00:00:00|aleatoria|     0|
|        1003| 8447.92|{Jonathan Gonsalv...|{Maria Fernanda C...|2022-07-04 00:00:00|aleatoria|     0|
|        1004|   58.51|{Jonathan Gonsalv...|{Isabel Silva, C6...|2021-09-11 00:00:00|aleatoria|     0|
|        1005| 6655.12|{Jonathan Gonsalv...|{Anthony Carvalho...|2022-02-11 00:00:00|  celular|     0|
|        1006| 9912.25|{Jonathan Gonsalv...|{Eloah Monteiro, ...|2022-05-

In [30]:
df.select('destinatario.nome').show()

+--------------------+
|                nome|
+--------------------+
|       Gabriel Cunha|
|         Diego Souza|
|        Nicole Nunes|
|Maria Fernanda Ca...|
|        Isabel Silva|
|    Anthony Carvalho|
|      Eloah Monteiro|
|        Sophie Rocha|
|      Pietro Ribeiro|
|      Eloah Teixeira|
|     Emanuella Sales|
|    Valentina Campos|
|       Stella Araujo|
|     Benicio Costela|
|      Joao Fernandes|
|   Gabriela da Rocha|
|      Larissa Aragao|
|           Theo Dias|
|        Danilo Jesus|
|       Bruno Correia|
+--------------------+
only showing top 20 rows


In [31]:
df.write.mode('overwrite').partitionBy('chave_pix').parquet('outpute/pix')

# SparkSQL

In [32]:
!pip install pyspark

In [33]:
import time

# Give ngrok some more time to establish the tunnel if it hasn't already
time.sleep(5)

# Retrieve and print the public URL for the Spark UI
ngrok_url = !curl -s http://localhost:4040/api/tunnels | grep -oP '(?<="public_url":")[^"]*'

if ngrok_url:
    print(f"Spark UI is available at: {ngrok_url[0]}")
else:
    print("Could not retrieve ngrok public URL. Please ensure ngrok is running.")

Could not retrieve ngrok public URL. Please ensure ngrok is running.


In [34]:
!wget -qnc https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
!unzip -n -q ngrok-stable-linux-amd64.zip

In [35]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
      .config('spark.ui.port', '4040')
      .appName("SparkUI Introdução")
      .getOrCreate()
)

In [36]:
# 1. Instala a ferramenta de conexão
!pip install pyngrok

from pyngrok import ngrok

# --- SEGURANÇA (COLE SEU TOKEN ABAIXO) ---
# Sem o token, o túnel vai cair rápido.
SEU_TOKEN = ""
ngrok.set_auth_token(SEU_TOKEN)

# 2. Mata processos velhos para limpar a área
ngrok.kill()

# 3. Descobre em qual porta o Spark está (Geralmente 4040, mas pode mudar)
try:
    # Pega a porta direto do SparkContext
    ui_url = spark.sparkContext.uiWebUrl
    port = ui_url.split(':')[-1]
    print(f"O Spark está rodando na porta interna: {port}")

    # 4. Abre o túnel para essa porta específica
    public_url = ngrok.connect(port).public_url
    print(f"\n🎉 CLIQUE AQUI PARA ABRIR O SPARK UI: {public_url}")

except Exception as e:
    print("Erro: O Spark não parece estar ativo. Rode 'spark = SparkSession.builder...' primeiro.")

O Spark está rodando na porta interna: 4040

🎉 CLIQUE AQUI PARA ABRIR O SPARK UI: https://cleopatra-arthritic-regerminatively.ngrok-free.dev


In [37]:
print("Checking ngrok tunnels API...")
!curl -s http://localhost:4040/api/tunnels

Checking ngrok tunnels API...
<html>
<head>
<meta http-equiv="Content-Type" content="text/html;charset=ISO-8859-1"/>
<title>Error 404 Not Found</title>
</head>
<body><h2>HTTP ERROR 404 Not Found</h2>
<table>
<tr><th>URI:</th><td>/api/tunnels</td></tr>
<tr><th>STATUS:</th><td>404</td></tr>
<tr><th>MESSAGE:</th><td>Not Found</td></tr>
<tr><th>SERVLET:</th><td>org.glassfish.jersey.servlet.ServletContainer-79e812d9</td></tr>
</table>

</body>
</html>


In [38]:
# {
#     "id_transacao": 1000,
#     "valor": "58931.97",
#     "remetente": {"nome": "Jonathan Gonsalves", "banco": "BTG", "tipo": "PF"},
#     "destinatario": {"nome": "Emanuella Moura", "banco": "Itau", "tipo": "PJ"},
#     "transaction_date": "2021-06-02",
#     "chave_pix": "aleatoria",
#     "fraude": "1"
# }

from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, TimestampType

schema_remetente_destinatario = StructType([
    StructField('nome', StringType()),
    StructField('banco', StringType()),
    StructField('tipo', StringType()),
])


schema_base_pix = StructType([
    StructField('id_transacao', IntegerType()),
    StructField('valor', DoubleType()),
    StructField('remetente', schema_remetente_destinatario),
    StructField('destinatario', schema_remetente_destinatario),
    StructField('transaction_date', TimestampType()),
    StructField('chave_pix', StringType()),
    StructField('fraude', IntegerType())
])


caminho_json = '/content/pix_transactions.json'

spark.read.json(
    caminho_json,
    schema=schema_base_pix,
    timestampFormat="yyyy-MM-dd"
).createOrReplaceTempView('transacoes_pix')


df = spark.read.json(
    caminho_json,
    schema=schema_base_pix,
    timestampFormat="yyyy-MM-dd"
)

In [39]:
spark.sql("select * from transacoes_pix limit 10").show()

+------------+-------+--------------------+--------------------+-------------------+---------+------+
|id_transacao|  valor|           remetente|        destinatario|   transaction_date|chave_pix|fraude|
+------------+-------+--------------------+--------------------+-------------------+---------+------+
|        1000|   7.05|{Jonathan Gonsalv...|{Gabriel Cunha, I...|2022-03-19 00:00:00|      cpf|     0|
|        1001|  37.28|{Jonathan Gonsalv...|{Diego Souza, XP,...|2021-01-26 00:00:00|aleatoria|     0|
|        1002| 282.73|{Jonathan Gonsalv...|{Nicole Nunes, BT...|2022-05-31 00:00:00|aleatoria|     0|
|        1003|8447.92|{Jonathan Gonsalv...|{Maria Fernanda C...|2022-07-04 00:00:00|aleatoria|     0|
|        1004|  58.51|{Jonathan Gonsalv...|{Isabel Silva, C6...|2021-09-11 00:00:00|aleatoria|     0|
|        1005|6655.12|{Jonathan Gonsalv...|{Anthony Carvalho...|2022-02-11 00:00:00|  celular|     0|
|        1006|9912.25|{Jonathan Gonsalv...|{Eloah Monteiro, ...|2022-05-10 00:00:0

In [40]:
group_sql = spark.sql('select chave_pix, count(*) from transacoes_pix group by chave_pix')

In [41]:
group_df = df.groupBy('chave_pix').count()

In [42]:
group_sql.explain()

group_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[chave_pix#56], functions=[count(1)])
   +- Exchange hashpartitioning(chave_pix#56, 200), ENSURE_REQUIREMENTS, [plan_id=74]
      +- HashAggregate(keys=[chave_pix#56], functions=[partial_count(1)])
         +- FileScan json [chave_pix#56] Batched: false, DataFilters: [], Format: JSON, Location: InMemoryFileIndex(1 paths)[file:/content/pix_transactions.json], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<chave_pix:string>


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[chave_pix#63], functions=[count(1)])
   +- Exchange hashpartitioning(chave_pix#63, 200), ENSURE_REQUIREMENTS, [plan_id=87]
      +- HashAggregate(keys=[chave_pix#63], functions=[partial_count(1)])
         +- FileScan json [chave_pix#63] Batched: false, DataFilters: [], Format: JSON, Location: InMemoryFileIndex(1 paths)[file:/content/pix_transactions.json], PartitionFilters: [], PushedFilters: [], R

In [43]:
group_sql.show()

+---------+--------+
|chave_pix|count(1)|
+---------+--------+
|aleatoria|   25045|
|  celular|   24841|
|    email|   24935|
|      cpf|   25179|
+---------+--------+



In [44]:
group_df.show()

+---------+-----+
|chave_pix|count|
+---------+-----+
|aleatoria|25045|
|  celular|24841|
|    email|24935|
|      cpf|25179|
+---------+-----+



In [45]:
spark.sql("""
    select
        chave_pix,
        round(avg(valor), 3)
    from transacoes_pix
    group by 1
    order by 2 desc
""").show()

+---------+--------------------+
|chave_pix|round(avg(valor), 3)|
+---------+--------------------+
|aleatoria|           12217.234|
|  celular|            12152.68|
|      cpf|           11946.072|
|    email|           11868.017|
+---------+--------------------+



In [46]:
spark.sql("""
    select
        chave_pix,
        count(*) as count_maior_100
    from transacoes_pix
    where valor > 10000
    group by 1
    order by 1 desc
""").show()

+---------+---------------+
|chave_pix|count_maior_100|
+---------+---------------+
|    email|           4830|
|      cpf|           4950|
|  celular|           4922|
|aleatoria|           5032|
+---------+---------------+



In [47]:
spark.sql("""
    with cte_base_window as (
        select
            destinatario.banco,
            valor,
            row_number() over (partition by destinatario.banco order by valor desc) as row_number
        from transacoes_pix
    )
    select
        banco,
        valor
    from cte_base_window
    where row_number in (1, 2)
""").show()

+--------+--------+
|   banco|   valor|
+--------+--------+
|     BTG|99946.78|
|     BTG| 99913.9|
|Bradesco|99910.87|
|Bradesco|99887.88|
|      C6|99980.03|
|      C6|99964.99|
|   Caixa|99969.06|
|   Caixa|99933.09|
|    Itau|99999.54|
|    Itau|99951.02|
|  Nubank|99935.45|
|  Nubank|99914.35|
|      XP|99961.28|
|      XP|99934.01|
+--------+--------+



In [48]:
df_row_number = spark.sql("""
    select
        destinatario.banco,
        valor,
        row_number() over (partition by destinatario.banco order by valor desc) as row_number
    from transacoes_pix
""")

In [49]:
df_row_number.show()

+-----+--------+----------+
|banco|   valor|row_number|
+-----+--------+----------+
|  BTG|99946.78|         1|
|  BTG| 99913.9|         2|
|  BTG|99873.58|         3|
|  BTG|99865.12|         4|
|  BTG|99840.68|         5|
|  BTG|99832.08|         6|
|  BTG| 99829.9|         7|
|  BTG|99814.23|         8|
|  BTG|99813.42|         9|
|  BTG|99785.91|        10|
|  BTG|99754.22|        11|
|  BTG|99750.69|        12|
|  BTG|99724.27|        13|
|  BTG|99711.66|        14|
|  BTG|99708.06|        15|
|  BTG|99684.07|        16|
|  BTG|99677.36|        17|
|  BTG|99648.38|        18|
|  BTG|99635.23|        19|
|  BTG|99628.33|        20|
+-----+--------+----------+
only showing top 20 rows


In [50]:
from pyspark.sql.functions import col

df_row_number.filter(col('row_number').isin([1,2])).show()

+--------+--------+----------+
|   banco|   valor|row_number|
+--------+--------+----------+
|     BTG|99946.78|         1|
|     BTG| 99913.9|         2|
|Bradesco|99910.87|         1|
|Bradesco|99887.88|         2|
|      C6|99980.03|         1|
|      C6|99964.99|         2|
|   Caixa|99969.06|         1|
|   Caixa|99933.09|         2|
|    Itau|99999.54|         1|
|    Itau|99951.02|         2|
|  Nubank|99935.45|         1|
|  Nubank|99914.35|         2|
|      XP|99961.28|         1|
|      XP|99934.01|         2|
+--------+--------+----------+

